In [ ]:
# 数据拷贝
import shutil

# 新建文件夹
os.makedirs("./ICC/data/A", exist_ok=True)
os.makedirs("./ICC/data/V", exist_ok=True)

for file in os.listdir("./ICC/data/inter/DOT/DOT1/"):
    print(file)
    shutil.copyfile("./data/crop_A/" + file, "./ICC/data/A/" + file)
    shutil.copyfile("./data/crop_V/" + file, "./ICC/data/V/" + file)

In [ ]:
# -*- coding: utf-8 -*-
"""
胰腺肿瘤异质性分析完整流程 v5.0 (patched)
- 先验知识学习（从标注数据）
- ConstrainedKMeans（内置5%约束）
- 强制3类 + HU梯度排序（基于Raw HU）
"""

import numpy as np
import pandas as pd
import nibabel as nib
from pathlib import Path
import warnings
import pickle
import os

from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import RobustScaler
from scipy.ndimage import generic_filter, sobel
from scipy.stats import norm, gaussian_kde
from scipy.spatial.distance import cdist

warnings.filterwarnings("ignore")


# ============================================================================
#  ConstrainedKMeans - 带最小簇大小约束的KMeans
# ============================================================================

class ConstrainedKMeans:
    """带最小簇大小约束的KMeans（贪心约束分配 + 迭代更新）"""

    def __init__(self, n_clusters=3, min_cluster_fraction=0.05, max_iter=100,
                 random_state=42, init="k-means++"):
        self.n_clusters = n_clusters
        self.min_cluster_fraction = min_cluster_fraction
        self.max_iter = max_iter
        self.random_state = random_state
        self.init = init
        self.cluster_centers_ = None
        self.labels_ = None

    def fit_predict(self, X, init_centers=None):
        np.random.seed(self.random_state)
        n_samples = len(X)
        min_size = int(np.ceil(n_samples * self.min_cluster_fraction))

        # 初始化中心
        if init_centers is not None and init_centers.shape[0] == self.n_clusters:
            centers = init_centers.copy()
            print("    使用自定义中心初始化")
        elif self.init == "k-means++":
            kmeans = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=1, init="k-means++")
            kmeans.fit(X)
            centers = kmeans.cluster_centers_
        else:
            kmeans = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=10)
            kmeans.fit(X)
            centers = kmeans.cluster_centers_

        # 初始约束分配
        labels = self._constrained_assign(X, centers, min_size)
        centers = self._update_centers(X, labels)

        # 迭代
        for _ in range(self.max_iter):
            unique_labels, counts = np.unique(labels, return_counts=True)
            if len(unique_labels) != self.n_clusters:
                labels = self._constrained_assign(X, centers, min_size)
                centers = self._update_centers(X, labels)
                continue

            new_labels = self._assign_labels(X, centers)
            new_unique, new_counts = np.unique(new_labels, return_counts=True)

            if len(new_unique) == self.n_clusters and np.all(new_counts >= min_size):
                new_centers = self._update_centers(X, new_labels)
                if np.allclose(centers, new_centers, rtol=1e-4):
                    labels, centers = new_labels, new_centers
                    break
                labels, centers = new_labels, new_centers
            else:
                labels = self._constrained_assign(X, centers, min_size)
                centers = self._update_centers(X, labels)

        self.cluster_centers_ = centers
        self.labels_ = labels
        return labels

    def _assign_labels(self, X, centers):
        distances = cdist(X, centers, metric="euclidean")
        return np.argmin(distances, axis=1)

    def _constrained_assign(self, X, centers, min_size):
        n_samples = len(X)
        distances = cdist(X, centers, metric="euclidean")

        labels = np.full(n_samples, -1, dtype=int)
        cluster_sizes = np.zeros(self.n_clusters, dtype=int)

        sample_indices, cluster_indices = np.meshgrid(
            np.arange(n_samples), np.arange(self.n_clusters), indexing="ij"
        )
        flat_samples = sample_indices.ravel()
        flat_clusters = cluster_indices.ravel()
        flat_distances = distances.ravel()

        sorted_idx = np.argsort(flat_distances)

        assigned = set()

        # 第一阶段：保证每簇至少 min_size
        for idx in sorted_idx:
            s = int(flat_samples[idx])
            c = int(flat_clusters[idx])
            if s in assigned:
                continue
            if cluster_sizes[c] < min_size:
                labels[s] = c
                cluster_sizes[c] += 1
                assigned.add(s)
                if np.all(cluster_sizes >= min_size):
                    break

        # 第二阶段：剩余样本按最近中心
        unassigned = [i for i in range(n_samples) if i not in assigned]
        for s in unassigned:
            c = int(np.argmin(distances[s]))
            labels[s] = c
            cluster_sizes[c] += 1

        # 兜底
        if np.any(labels == -1):
            mask = labels == -1
            labels[mask] = np.argmin(distances[mask], axis=1)

        return labels

    def _update_centers(self, X, labels):
        centers = np.zeros((self.n_clusters, X.shape[1]), dtype=float)
        for k in range(self.n_clusters):
            mask = labels == k
            if np.any(mask):
                centers[k] = X[mask].mean(axis=0)
            else:
                centers[k] = X[np.random.randint(len(X))]
        return centers


# ============================================================================
#  TumorHeterogeneityPriorKnowledge - 先验知识学习器
# ============================================================================

class TumorHeterogeneityPriorKnowledge:
    """肿瘤异质性先验知识学习器（基于Raw HU的动脉期三维特征）"""

    def __init__(self, min_cluster_fraction=0.05):
        self.prior_learned = False
        self.inter_tumor_stats = {}
        self.intra_tumor_stats = {}
        self.feature_distributions = {}
        self.cluster_centroids = {}
        self.feature_importance = {}
        self.feature_dimension = 3
        self.min_cluster_fraction = min_cluster_fraction
        self.n_clusters = 3

        # 新增：先验尺度（用于 similarity 归一）
        self.prior_center_scale = None  # shape=(d,)

    def learn_from_annotations(self, all_analyzers):
        print("\n🔬 从临床标注点学习先验知识...")

        all_annotation_data = {1: [], 2: [], 3: []}
        intra_tumor_differences = []

        for analyzer in all_analyzers:
            if not analyzer.has_annotations:
                continue

            # 使用 Raw HU 的动脉期特征（3维）
            features_a = analyzer.extract_arterial_features(use_raw_hu=True)
            self.feature_dimension = features_a.shape[1]

            tumor_class_hus = {1: [], 2: [], 3: []}

            for ann_idx, label in zip(analyzer.annotation_indices, analyzer.annotation_labels):
                all_annotation_data[int(label)].append(features_a[int(ann_idx)])
                tumor_class_hus[int(label)].append(float(features_a[int(ann_idx), 0]))  # raw weighted HU (still HU-scale)

            intra_diffs = self._compute_intra_tumor_differences(tumor_class_hus)
            if intra_diffs:
                intra_tumor_differences.append(intra_diffs)

        total_annotations = sum(len(v) for v in all_annotation_data.values())
        if total_annotations < 10:
            print("⚠️  标注点不足，无法学习有效先验知识")
            return False

        print(f"  总标注点: {total_annotations}")
        for label in [1, 2, 3]:
            print(f"    Label{label}: {len(all_annotation_data[label])}")

        self._learn_intra_tumor_patterns(intra_tumor_differences)
        self._learn_inter_tumor_distribution(all_annotation_data)
        self._learn_feature_distributions(all_annotation_data)
        self._learn_cluster_centroids(all_annotation_data)
        self._compute_feature_importance(all_annotation_data)
        self._learn_prior_scale(all_annotation_data)

        self.prior_learned = True
        print("✅ 先验知识学习完成")
        return True

    def _learn_prior_scale(self, annotation_data):
        """保存每维 robust scale (IQR)，用于 similarity 归一化"""
        all_feats = []
        for label in [1, 2, 3]:
            if len(annotation_data[label]) > 0:
                all_feats.append(np.array(annotation_data[label], dtype=float))
        if not all_feats:
            return
        X = np.vstack(all_feats)
        q75 = np.percentile(X, 75, axis=0)
        q25 = np.percentile(X, 25, axis=0)
        iqr = q75 - q25
        iqr[iqr < 1e-6] = 1.0
        self.prior_center_scale = iqr

    def _compute_intra_tumor_differences(self, tumor_class_hus):
        diffs = {}
        valid_classes = [k for k, v in tumor_class_hus.items() if len(v) > 0]
        if len(valid_classes) < 2:
            return None

        for i, ci in enumerate(valid_classes):
            for cj in valid_classes[i + 1:]:
                mean_i = np.mean(tumor_class_hus[ci])
                mean_j = np.mean(tumor_class_hus[cj])
                diffs[f"class_{ci}_vs_{cj}"] = abs(mean_j - mean_i)

        for ck in valid_classes:
            if len(tumor_class_hus[ck]) > 1:
                diffs[f"class_{ck}_intra_std"] = np.std(tumor_class_hus[ck])

        return diffs

    def _learn_intra_tumor_patterns(self, intra_tumor_differences):
        if len(intra_tumor_differences) == 0:
            return

        all_diffs = {}
        for tumor_diffs in intra_tumor_differences:
            for key, value in tumor_diffs.items():
                all_diffs.setdefault(key, []).append(value)

        self.intra_tumor_stats = {}
        for key, values in all_diffs.items():
            self.intra_tumor_stats[key] = {
                "mean": float(np.mean(values)),
                "std": float(np.std(values)),
                "median": float(np.median(values)),
                "q25": float(np.percentile(values, 25)),
                "q75": float(np.percentile(values, 75)),
            }

        print("\n  📊 肿瘤内差异先验统计:")
        for key, stats in sorted(self.intra_tumor_stats.items()):
            if "vs" in key:
                print(f"    {key}: {stats['median']:.2f} [{stats['q25']:.2f}, {stats['q75']:.2f}]")

    def _learn_inter_tumor_distribution(self, annotation_data):
        for label in [1, 2, 3]:
            if len(annotation_data[label]) == 0:
                continue
            data = np.array(annotation_data[label], dtype=float)
            self.inter_tumor_stats[label] = {
                "mean_a_hu": float(np.mean(data[:, 0])),
                "std_a_hu": float(np.std(data[:, 0])),
                "median_a_hu": float(np.median(data[:, 0])),
            }

    def _learn_feature_distributions(self, annotation_data):
        feature_names = ["weighted_a_hu", "a_texture", "a_gradient"]
        for label in [1, 2, 3]:
            if len(annotation_data[label]) == 0:
                continue
            data = np.array(annotation_data[label], dtype=float)
            self.feature_distributions[label] = {}
            for i, name in enumerate(feature_names[: data.shape[1]]):
                if len(data[:, i]) > 1:
                    try:
                        self.feature_distributions[label][name] = gaussian_kde(data[:, i])
                    except Exception:
                        mu, sigma = norm.fit(data[:, i])
                        sigma = float(max(sigma, 1e-6))
                        self.feature_distributions[label][name] = (float(mu), sigma)

    def _learn_cluster_centroids(self, annotation_data):
        centroids_by_label = {}
        for label in [1, 2, 3]:
            if len(annotation_data[label]) > 0:
                centroids_by_label[label] = np.median(np.array(annotation_data[label], dtype=float), axis=0)

        if len(centroids_by_label) == 3:
            sorted_labels = sorted(centroids_by_label.keys(), key=lambda x: centroids_by_label[x][0])
            self.cluster_centroids = {i + 1: centroids_by_label[sorted_labels[i]] for i in range(3)}

            print(f"\n  🎯 先验中心按HU排序: {sorted_labels} -> [1, 2, 3]")
            for i in range(3):
                print(f"    Label{i+1}: HU={self.cluster_centroids[i+1][0]:.2f}")
        else:
            self.cluster_centroids = centroids_by_label

    def _compute_feature_importance(self, annotation_data):
        all_features, all_labels = [], []
        for label in [1, 2, 3]:
            if len(annotation_data[label]) > 0:
                data = np.array(annotation_data[label], dtype=float)
                all_features.append(data)
                all_labels.extend([label] * len(data))

        if not all_features:
            return

        X = np.vstack(all_features)
        y = np.array(all_labels, dtype=int)
        n_features = X.shape[1]
        feature_importance = np.zeros(n_features, dtype=float)

        for i in range(n_features):
            feature_values = X[:, i]
            overall_mean = np.mean(feature_values)

            between_var = 0.0
            within_var = 0.0
            for label in [1, 2, 3]:
                m = y == label
                if not np.any(m):
                    continue
                n = np.sum(m)
                mu = np.mean(feature_values[m])
                between_var += n * (mu - overall_mean) ** 2
                within_var += np.sum((feature_values[m] - mu) ** 2)

            feature_importance[i] = (between_var / within_var) if within_var > 0 else 0.0

        if np.sum(feature_importance) > 0:
            feature_importance = feature_importance / np.sum(feature_importance)

        self.feature_importance = feature_importance
        names = ["A_HU", "A_texture", "A_gradient"][:n_features]
        print(f"\n  特征重要性: {dict(zip(names, feature_importance.tolist()))}")

    def get_prior_probabilities(self, features_vec, current_labels=None, n_total=None):
        """features_vec: 单个样本的(3,)动脉期特征（raw HU语义）"""
        if not self.prior_learned:
            return np.array([0.33, 0.33, 0.34], dtype=float)

        features_vec = np.asarray(features_vec, dtype=float)
        d = len(features_vec)

        probs = []
        for label in [1, 2, 3]:
            # similarity（用IQR归一化距离）
            similarity = 0.5
            if label in self.cluster_centroids:
                centroid = np.asarray(self.cluster_centroids[label], dtype=float)[:d]
                weights = self.feature_importance[:d] if len(self.feature_importance) >= d else np.ones(d, dtype=float)

                scale = self.prior_center_scale[:d] if self.prior_center_scale is not None else np.ones(d, dtype=float)
                z = (features_vec[:d] - centroid) / scale
                distance = np.sqrt(np.sum(weights * (z ** 2)))
                similarity = float(np.exp(-0.5 * distance))  # (0,1]

            # 分布概率
            prob_dist = 1.0
            if label in self.feature_distributions:
                for i, name in enumerate(["weighted_a_hu", "a_texture", "a_gradient"][:d]):
                    if name not in self.feature_distributions[label]:
                        continue
                    dist = self.feature_distributions[label][name]
                    if isinstance(dist, gaussian_kde):
                        prob_dist *= float(max(dist(features_vec[i])[0], 1e-10))
                    else:
                        mu, sigma = dist
                        prob_dist *= float(norm.pdf(features_vec[i], mu, sigma) + 1e-10)

            cluster_penalty = 1.0
            if current_labels is not None and n_total is not None:
                current_count = np.sum(current_labels == label)
                current_fraction = current_count / max(n_total, 1)
                if current_fraction < self.min_cluster_fraction:
                    cluster_penalty = 1.5

            probs.append((0.7 * prob_dist + 0.3 * similarity) * cluster_penalty)

        probs = np.array(probs, dtype=float)
        return probs / (np.sum(probs) + 1e-12)

    def get_intra_tumor_consistency_score(self, features, labels):
        if not self.intra_tumor_stats:
            return 1.0

        score = 1.0
        unique_labels = np.unique(labels)

        for i, li in enumerate(unique_labels):
            for lj in unique_labels[i + 1:]:
                mi = labels == li
                mj = labels == lj
                if np.sum(mi) == 0 or np.sum(mj) == 0:
                    continue

                mean_i = np.mean(features[mi, 0])
                mean_j = np.mean(features[mj, 0])
                observed_diff = abs(mean_j - mean_i)

                diff_key = f"class_{min(li, lj)}_vs_{max(li, lj)}"
                if diff_key not in self.intra_tumor_stats:
                    continue

                stats = self.intra_tumor_stats[diff_key]
                expected_median = stats["median"]
                q25, q75 = stats["q25"], stats["q75"]

                if q25 <= observed_diff <= q75:
                    score *= 1.1
                else:
                    deviation = min(abs(observed_diff - q25), abs(observed_diff - q75)) / max(expected_median, 1e-6)
                    score *= max(0.8, 1.0 - 0.1 * deviation)

        return float(np.clip(score, 0.5, 1.5))

    def save_prior_knowledge(self, filepath):
        with open(filepath, "wb") as f:
            pickle.dump({
                "inter_tumor_stats": self.inter_tumor_stats,
                "intra_tumor_stats": self.intra_tumor_stats,
                "feature_distributions": self.feature_distributions,
                "cluster_centroids": self.cluster_centroids,
                "feature_importance": self.feature_importance,
                "prior_learned": self.prior_learned,
                "feature_dimension": self.feature_dimension,
                "min_cluster_fraction": self.min_cluster_fraction,
                "n_clusters": self.n_clusters,
                "prior_center_scale": self.prior_center_scale,
            }, f)
        print(f"\n💾 先验知识已保存: {filepath}")

    def load_prior_knowledge(self, filepath):
        try:
            with open(filepath, "rb") as f:
                data = pickle.load(f)
            self.__dict__.update(data)
            print(f"\n📂 先验知识已加载: {filepath}")
            return True
        except Exception as e:
            print(f"❌ 加载失败: {e}")
            return False


# ============================================================================
#  PancreaticTumorHeterogeneityAnalyzer - 肿瘤分析器
# ============================================================================

class PancreaticTumorHeterogeneityAnalyzer:
    """胰腺癌异质性分析器"""

    def __init__(self, arterial_ct_raw, venous_ct_raw, tumor_mask, doctor_annotations=None,
                 arterial_weight=2.0, prior_knowledge=None, min_cluster_fraction=0.05):
        self.arterial_raw = np.asarray(arterial_ct_raw, dtype=np.float32)
        self.venous_raw = np.asarray(venous_ct_raw, dtype=np.float32)

        self.mask = tumor_mask.astype(bool)
        self.arterial_weight = float(arterial_weight)
        self.prior_knowledge = prior_knowledge
        self.min_cluster_fraction = float(min_cluster_fraction)
        self.n_clusters = 3

        self.annotation_points = None
        self.annotation_labels = None
        self.annotation_indices = None
        self.has_annotations = False
        self.labels = None

        if doctor_annotations is not None:
            self._parse_doctor_annotations(doctor_annotations)

    def _parse_doctor_annotations(self, doctor_annotations):
        """解析医生标注（3D中 label=1/2/3）"""
        try:
            ann = np.asarray(doctor_annotations)
            coords = np.where(ann > 0)
            if len(coords[0]) == 0:
                return

            points = np.column_stack(coords)
            labels = ann[coords].astype(int)

            valid_points, valid_labels = [], []
            for p, lb in zip(points, labels):
                if self.mask[tuple(p)] and lb in [1, 2, 3]:
                    valid_points.append(p)
                    valid_labels.append(lb)

            if len(valid_points) == 0:
                return

            self.annotation_points = np.array(valid_points, dtype=int)
            self.annotation_labels = np.array(valid_labels, dtype=int)

            mask_coords = np.array(np.where(self.mask)).T  # (N,3)
            self.annotation_indices = [
                int(np.argmin(np.sum((mask_coords - p) ** 2, axis=1)))
                for p in self.annotation_points
            ]

            self.has_annotations = True
        except Exception:
            self.has_annotations = False

    def _arterial_std3(self):
        return generic_filter(self.arterial_raw, np.std, size=3)

    def _arterial_gradmag(self):
        return np.sqrt(sum(sobel(self.arterial_raw, axis=i) ** 2 for i in range(3)))

    def extract_arterial_features(self, use_raw_hu=True):
        """动脉期三维特征：weighted_a_hu, texture(std3), gradient"""
        A = self.arterial_raw if use_raw_hu else self.arterial_norm()
        tumor_voxels_a = A[self.mask]

        return np.column_stack([
            tumor_voxels_a * self.arterial_weight,
            self._arterial_std3()[self.mask],
            self._arterial_gradmag()[self.mask],
        ]).astype(np.float64)

    def arterial_norm(self):
        """仅在需要时使用归一化强度（不会覆盖 raw）"""
        A = self.arterial_raw
        return np.clip((A + 160.0) / 400.0, 0.0, 1.0).astype(np.float32)

    def venous_norm(self):
        V = self.venous_raw
        return np.clip((V + 160.0) / 400.0, 0.0, 1.0).astype(np.float32)

    def extract_combined_features(self, use_raw_hu=True):
        """组合特征（默认用 raw HU 语义；若你希望用归一化，可传 False）"""
        A = self.arterial_raw if use_raw_hu else self.arterial_norm()
        V = self.venous_raw if use_raw_hu else self.venous_norm()

        tumor_voxels_a = A[self.mask]
        tumor_voxels_v = V[self.mask]

        weighted_a = tumor_voxels_a * self.arterial_weight
        enhancement_diff = tumor_voxels_v - tumor_voxels_a

        return np.column_stack([
            weighted_a,
            self._arterial_std3()[self.mask],
            self._arterial_gradmag()[self.mask],
            enhancement_diff,
            tumor_voxels_v / (tumor_voxels_a + 1e-6),
            np.abs(enhancement_diff),
            tumor_voxels_v,
            (weighted_a + tumor_voxels_v) / 2.0,
            np.where(enhancement_diff > 0.05, 1, np.where(enhancement_diff < -0.05, -1, 0)),
        ]).astype(np.float64)

    def adaptive_clustering(self, features=None, method="prior_guided", hu_for_sort=None):
        """返回 1..3 标签"""
        if features is None:
            features = self.extract_combined_features(use_raw_hu=True)

        # HU梯度排序的依据：默认用 raw weighted_a_hu（features[:,0] 已是 raw*weight）
        if hu_for_sort is None:
            hu_for_sort = features[:, 0]

        print(f"\n🔬 聚类方法: {method}, 特征维度: {features.shape}")

        if method == "prior_guided" and self.prior_knowledge and self.prior_knowledge.prior_learned:
            labels0 = self._prior_guided_clustering(features, hu_for_sort=hu_for_sort)
        elif self.has_annotations:
            labels0 = self._annotation_guided_clustering(features, hu_for_sort=hu_for_sort)
        else:
            labels0 = self._basic_clustering(features, hu_for_sort=hu_for_sort)

        # 验证与修正（0..2）
        unique_labels, counts = np.unique(labels0, return_counts=True)
        n_samples = len(labels0)
        min_size = int(np.ceil(n_samples * self.min_cluster_fraction))

        print("\n  📊 聚类结果(初始):")
        for lb, ct in zip(unique_labels, counts):
            status = "✓" if ct >= min_size else "✗"
            print(f"    簇{lb}: {ct:5d} ({ct/n_samples*100:5.1f}%) [{status}]")

        if len(unique_labels) != self.n_clusters or np.any(counts < min_size):
            print("  ⚠️  约束未满足，启动后处理...")
            labels0 = self._ensure_exactly_n_clusters(labels0, features)
            labels0 = self._enforce_minimum_cluster_size_postprocess(labels0, features)
            labels0 = self._ensure_exactly_n_clusters(labels0, features)

        # HU梯度排序（基于 raw HU 语义）
        labels0 = self._ensure_hu_gradient(labels0, hu_for_sort)

        # 一致性评估（用动脉期raw三维特征）
        if self.prior_knowledge and self.prior_knowledge.prior_learned:
            a3 = features[:, :3]
            score = self.prior_knowledge.get_intra_tumor_consistency_score(a3, labels0 + 1)  # prior里用1..3统计key
            print(f"  🎯 肿瘤内差异一致性: {score:.3f}")

        return (labels0.astype(int) + 1)

    def _prior_guided_clustering(self, features, hu_for_sort):
        a_features = features[:, :3]
        av_features = features[:, 3:]

        # 动脉期特征选择（按 prior feature importance）
        if hasattr(self.prior_knowledge, "feature_importance") and len(self.prior_knowledge.feature_importance) > 0:
            imp = np.asarray(self.prior_knowledge.feature_importance, dtype=float)
            selected_indices = np.where(imp[:a_features.shape[1]] > np.mean(imp[:a_features.shape[1]]) * 0.8)[0]
            if len(selected_indices) == 0:
                selected_indices = np.arange(a_features.shape[1])
        else:
            selected_indices = np.arange(a_features.shape[1])

        a_processed = RobustScaler().fit_transform(a_features[:, selected_indices])

        av_variances = np.var(av_features, axis=0)
        av_mask = av_variances > np.percentile(av_variances, 30)
        av_processed = RobustScaler().fit_transform(av_features[:, av_mask]) if np.any(av_mask) else np.zeros((len(av_features), 0))

        combined = np.hstack([a_processed * 0.6, av_processed * 0.4])

        # 先验中心初始化（动脉期部分可用，AV部分置0）
        init_centers = None
        if len(getattr(self.prior_knowledge, "cluster_centroids", {})) == 3:
            try:
                prior_centers_raw = [
                    np.asarray(self.prior_knowledge.cluster_centroids[i + 1], dtype=float)[:len(selected_indices)]
                    for i in range(3)
                ]
                scaler_a = RobustScaler().fit(a_features[:, selected_indices])
                prior_a_scaled = scaler_a.transform(prior_centers_raw)
                prior_av_scaled = np.zeros((3, av_processed.shape[1]), dtype=float)
                init_centers = np.hstack([prior_a_scaled * 0.6, prior_av_scaled * 0.4])
                print("  🎯 使用先验中心初始化")
            except Exception:
                init_centers = None

        # ConstrainedKMeans → 回退GMM → 回退KMeans
        try:
            ck = ConstrainedKMeans(
                n_clusters=self.n_clusters,
                min_cluster_fraction=self.min_cluster_fraction,
                max_iter=100,
                random_state=42,
            )
            labels = ck.fit_predict(combined, init_centers=init_centers)
            print(f"  ✓ ConstrainedKMeans(n={self.n_clusters}, min≥{self.min_cluster_fraction*100:.0f}%)")
        except Exception as e:
            print(f"  ⚠️  ConstrainedKMeans失败: {e}, 回退到GMM")
            try:
                if init_centers is not None:
                    gmm = GaussianMixture(
                        n_components=self.n_clusters,
                        random_state=42,
                        max_iter=100,
                        n_init=1,
                        means_init=init_centers,
                    )
                else:
                    gmm = GaussianMixture(
                        n_components=self.n_clusters,
                        random_state=42,
                        max_iter=100,
                        n_init=3,
                    )
                gmm.fit(combined)
                post = gmm.predict_proba(combined)

                prior_probs = np.array([
                    self.prior_knowledge.get_prior_probabilities(a_features[i])
                    for i in range(len(a_features))
                ], dtype=float)

                comb = 0.65 * post + 0.35 * prior_probs
                labels = np.argmax(comb, axis=1)
                print("  ✓ GMM(n=3) + 先验概率")
            except Exception as e2:
                print(f"  ⚠️  GMM也失败: {e2}, 使用标准KMeans")
                labels = self._basic_clustering_impl(combined, init_centers)

        return self._reorder_labels_by_hu(labels, hu_for_sort)

    def _annotation_guided_clustering(self, features, hu_for_sort):
        X_scaled = RobustScaler().fit_transform(features)

        init_centers = None
        if self.has_annotations and len(np.unique(self.annotation_labels)) == 3:
            try:
                centers = []
                for label in [1, 2, 3]:
                    m = self.annotation_labels == label
                    if np.any(m):
                        idx = np.array(self.annotation_indices, dtype=int)[m]
                        centers.append(np.mean(X_scaled[idx], axis=0))
                if len(centers) == 3:
                    init_centers = np.array(centers, dtype=float)

                    # 修复：用 annotation_indices 取对应肿瘤体素的HU均值排序
                    hu_means = []
                    for label in [1, 2, 3]:
                        m = self.annotation_labels == label
                        idx = np.array(self.annotation_indices, dtype=int)[m]
                        hu_means.append(np.mean(hu_for_sort[idx]) if len(idx) else np.inf)
                    sorted_idx = np.argsort(hu_means)
                    init_centers = init_centers[sorted_idx]
                    print("  🎯 使用标注中心初始化")
            except Exception:
                init_centers = None

        try:
            ck = ConstrainedKMeans(
                n_clusters=self.n_clusters,
                min_cluster_fraction=self.min_cluster_fraction,
                max_iter=100,
                random_state=42,
            )
            labels = ck.fit_predict(X_scaled, init_centers=init_centers)
            print("  ✓ ConstrainedKMeans + 标注")
        except Exception as e:
            print(f"  ⚠️  失败: {e}, 回退")
            labels = self._basic_clustering_impl(X_scaled, init_centers)

        return self._reorder_labels_by_hu(labels, hu_for_sort)

    def _basic_clustering(self, features, hu_for_sort):
        X_scaled = RobustScaler().fit_transform(features)
        labels = self._basic_clustering_impl(X_scaled, init_centers=None)
        return self._reorder_labels_by_hu(labels, hu_for_sort)

    def _basic_clustering_impl(self, X_scaled, init_centers=None):
        try:
            ck = ConstrainedKMeans(
                n_clusters=self.n_clusters,
                min_cluster_fraction=self.min_cluster_fraction,
                max_iter=100,
                random_state=42,
            )
            labels = ck.fit_predict(X_scaled, init_centers=init_centers)
            print("  ✓ ConstrainedKMeans")
            return labels
        except Exception as e:
            print(f"  ⚠️  失败: {e}, 使用标准KMeans")
            if init_centers is not None:
                kmeans = KMeans(n_clusters=self.n_clusters, init=init_centers, n_init=1, random_state=42)
            else:
                kmeans = KMeans(n_clusters=self.n_clusters, random_state=42, n_init=10)
            return kmeans.fit_predict(X_scaled)

    def _reorder_labels_by_hu(self, labels, hu_values):
        unique_labels = np.unique(labels)
        hu_means = {lb: float(np.mean(hu_values[labels == lb])) for lb in unique_labels}
        sorted_labels = sorted(unique_labels, key=lambda x: hu_means[x])
        mapping = {sorted_labels[i]: i for i in range(len(sorted_labels))}
        new_labels = np.vectorize(mapping.get)(labels).astype(int)

        print("  🔄 按HU值排序: " +
              f"{[f'L{old}({hu_means[old]:.2f})' for old in sorted_labels]} -> [0,1,2]")
        return new_labels

    def _ensure_hu_gradient(self, labels, hu_values):
        unique_labels = np.unique(labels)
        if len(unique_labels) != 3:
            return labels

        hu_means = {lb: float(np.mean(hu_values[labels == lb])) for lb in unique_labels}
        sorted_labels = sorted(unique_labels, key=lambda x: hu_means[x])

        if sorted_labels == [0, 1, 2]:
            print("  ✅ HU梯度已满足")
            return labels

        mapping = {sorted_labels[i]: i for i in range(3)}
        print("  🔄 HU梯度重排")
        return np.vectorize(mapping.get)(labels).astype(int)

    def _ensure_exactly_n_clusters(self, labels, features):
        unique_labels = np.unique(labels)
        n_current = len(unique_labels)
        if n_current == self.n_clusters:
            return labels

        print(f"  ⚠️  修正类别数: {n_current} -> {self.n_clusters}")

        if n_current < self.n_clusters:
            labels = self._split_clusters_to_target(labels, features, self.n_clusters)
        else:
            labels = self._merge_clusters_to_target(labels, features, self.n_clusters)
        return labels

    def _split_clusters_to_target(self, labels, features, target_n):
        unique_labels, counts = np.unique(labels, return_counts=True)
        n_missing = target_n - len(unique_labels)

        new_labels = labels.copy()
        next_label = int(np.max(unique_labels) + 1)

        for _ in range(n_missing):
            unique_labels, counts = np.unique(new_labels, return_counts=True)
            largest_label = unique_labels[np.argmax(counts)]
            mask = new_labels == largest_label
            X = features[mask]
            if len(X) < 2:
                break
            kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
            sub = kmeans.fit_predict(X)
            idx = np.where(mask)[0]
            new_labels[idx[sub == 1]] = next_label
            next_label += 1

        unique = np.unique(new_labels)
        mapping = {old: new for new, old in enumerate(unique)}
        return np.vectorize(mapping.get)(new_labels).astype(int)

    def _merge_clusters_to_target(self, labels, features, target_n):
        new_labels = labels.copy()
        while len(np.unique(new_labels)) > target_n:
            unique, counts = np.unique(new_labels, return_counts=True)
            smallest = unique[np.argmin(counts)]

            centers = {lb: np.mean(features[new_labels == lb], axis=0) for lb in unique}
            sc = centers[smallest]

            best_lb = None
            best_d = np.inf
            for lb in unique:
                if lb == smallest:
                    continue
                d = float(np.linalg.norm(centers[lb] - sc))
                if d < best_d:
                    best_d = d
                    best_lb = lb

            new_labels[new_labels == smallest] = best_lb

        unique = np.unique(new_labels)
        mapping = {old: new for new, old in enumerate(unique)}
        return np.vectorize(mapping.get)(new_labels).astype(int)

    def _enforce_minimum_cluster_size_postprocess(self, labels, features, max_iterations=5):
        n_samples = len(labels)
        min_size = int(np.ceil(n_samples * self.min_cluster_fraction))
        adjusted = labels.copy()

        for _ in range(max_iterations):
            unique, counts = np.unique(adjusted, return_counts=True)
            if len(unique) != self.n_clusters:
                break
            small = unique[counts < min_size]
            if len(small) == 0:
                break

            # 仍采用你的“整簇并入”策略（最小改动）；如需更稳，可再升级为搬运式
            for slb in small:
                m = adjusted == slb
                others = unique[unique != slb]
                if len(others) == 0:
                    break
                centers = {lb: np.mean(features[adjusted == lb], axis=0) for lb in unique}
                sc = centers[slb]
                dists = [np.linalg.norm(centers[olb] - sc) for olb in others]
                tgt = others[int(np.argmin(dists))]
                adjusted[np.where(m)[0]] = tgt
                break

        return adjusted

    def evaluate_clustering_quality(self, labels_1to3, hu_for_sort=None):
        labels = np.asarray(labels_1to3, dtype=int)
        unique, counts = np.unique(labels, return_counts=True)
        n_total = len(labels)

        features = self.extract_combined_features(use_raw_hu=True)
        if hu_for_sort is None:
            hu_for_sort = features[:, 0]

        hu_means = {int(lb): float(np.mean(hu_for_sort[labels == lb])) for lb in unique}
        ordered = sorted(unique.tolist(), key=lambda x: hu_means[int(x)])
        hu_gradient_correct = (ordered == [1, 2, 3])

        quality = {
            "n_clusters": int(len(unique)),
            "cluster_sizes": dict(zip(unique.tolist(), counts.tolist())),
            "size_balance": float(counts.min() / counts.max()) if len(counts) > 1 else 1.0,
            "min_cluster_fraction": float(counts.min() / max(n_total, 1)),
            "min_constraint_satisfied": (counts.min() / max(n_total, 1)) >= self.min_cluster_fraction,
            "exactly_3_clusters": len(unique) == 3,
            "hu_gradient_correct": bool(hu_gradient_correct),
            "hu_means": hu_means,
        }
        return quality

    def compute_tumor_descriptors(self, nii_img=None, labels_1to3=None):
        if labels_1to3 is None:
            if self.labels is None:
                raise ValueError("请先执行聚类!")
            labels = np.asarray(self.labels, dtype=int)
        else:
            labels = np.asarray(labels_1to3, dtype=int)

        voxel_vol = float(np.prod(nii_img.header.get_zooms()[:3])) if nii_img else 1.0
        features = self.extract_combined_features(use_raw_hu=True)

        descriptors = {
            "tumor_volume_mm3": float(np.sum(self.mask) * voxel_vol),
            "low_hu_fraction": float(np.sum(labels == 1) / len(labels)),
            "mid_hu_fraction": float(np.sum(labels == 2) / len(labels)),
            "high_hu_fraction": float(np.sum(labels == 3) / len(labels)),
            "mean_arterial_weighted_hu": float(np.mean(features[:, 0])),
            "mean_enhancement_diff": float(np.mean(features[:, 3])),
            "enhancement_heterogeneity": float(np.std(features[:, 3])),
        }

        for lb in [1, 2, 3]:
            m = labels == lb
            if np.any(m):
                descriptors[f"cluster_{lb}_mean_weighted_hu"] = float(np.mean(features[m, 0]))
                descriptors[f"cluster_{lb}_size"] = int(np.sum(m))
                descriptors[f"cluster_{lb}_mean_enhancement"] = float(np.mean(features[m, 3]))

        for i in [1, 2, 3]:
            for j in [i + 1, 2, 3]:
                mi = labels == i
                mj = labels == j
                if np.any(mi) and np.any(mj):
                    descriptors[f"intra_tumor_diff_class{i}_vs_{j}"] = float(abs(np.mean(features[mj, 0]) - np.mean(features[mi, 0])))

        return descriptors


# ============================================================================
#  工具函数
# ============================================================================

def load_nifti_data(a_path, v_path, roi_path, ann_path=None):
    """加载NIfTI数据（保留 raw HU；ROI阈值修正为>0）"""
    A_img = nib.load(str(a_path))
    V_img = nib.load(str(v_path))
    ROI_img = nib.load(str(roi_path))

    A_raw = A_img.get_fdata(dtype=np.float32)
    V_raw = V_img.get_fdata(dtype=np.float32)

    ROI = ROI_img.get_fdata()
    ROI = (ROI > 0)  # FIX: 原来 >1 会导致mask为空

    ANN = None
    if ann_path and Path(ann_path).exists():
        try:
            ANN_img = nib.load(str(ann_path))
            ANN = ANN_img.get_fdata()
        except Exception:
            ANN = None

    # 可选：提供归一化版本（不覆盖 raw）
    A_norm = np.clip((A_raw + 160.0) / 400.0, 0.0, 1.0).astype(np.float32)
    V_norm = np.clip((V_raw + 160.0) / 400.0, 0.0, 1.0).astype(np.float32)

    return A_img, V_img, ROI_img, A_raw, V_raw, ROI, ANN, A_norm, V_norm


def safe_output_name(name: str):
    """输出标签文件名：兼容 .nii / .nii.gz"""
    if name.endswith(".nii.gz"):
        stem = name[:-7]
        return stem + "_clusters.nii.gz"
    if name.endswith(".nii"):
        stem = name[:-4]
        return stem + "_clusters.nii"
    return name + "_clusters.nii.gz"


# ============================================================================
#  主要流程函数
# ============================================================================

def learn_prior_knowledge(data_dir, roi_dir, dot_dir,
                         arterial_weight=2.0, min_cluster_fraction=0.05,
                         save_path="./prior_knowledge.pkl"):
    print("\n" + "=" * 70)
    print("阶段1: 学习先验知识（强制3类 + HU梯度排序）")
    print("=" * 70)

    prior = TumorHeterogeneityPriorKnowledge(min_cluster_fraction=min_cluster_fraction)
    analyzers = []

    A_folder = os.path.join(data_dir, "A")
    V_folder = os.path.join(data_dir, "V")

    print("\n🔍 扫描标注数据...")
    for name in sorted(os.listdir(A_folder)):
        a_file = os.path.join(A_folder, name)
        v_file = os.path.join(V_folder, name)
        roi_file = os.path.join(roi_dir, name)
        ann_file = os.path.join(dot_dir, name)

        if not (Path(a_file).exists() and Path(v_file).exists() and Path(roi_file).exists()):
            continue

        _, _, _, A_raw, V_raw, ROI, ANN, _, _ = load_nifti_data(a_file, v_file, roi_file, ann_file)

        print("  ", name,
              "A_max:", float(np.max(A_raw)),
              "V_max:", float(np.max(V_raw)),
              "ROI_max:", float(np.max(ROI)),
              "ANN_max:", (float(np.max(ANN)) if ANN is not None else None))

        if np.sum(ROI) == 0:
            print(f"  ⚠️  ROI为空，跳过: {name}")
            continue

        analyzer = PancreaticTumorHeterogeneityAnalyzer(
            arterial_ct_raw=A_raw,
            venous_ct_raw=V_raw,
            tumor_mask=ROI,
            doctor_annotations=ANN,
            arterial_weight=arterial_weight,
            prior_knowledge=None,
            min_cluster_fraction=min_cluster_fraction,
        )
        if analyzer.has_annotations:
            analyzers.append(analyzer)
            print(f"  ✓ {name}: {len(analyzer.annotation_labels)}个标注")
        else:
            print(f"  ⏭️  {name}: 无有效标注点")

    print(f"\n✅ 共找到 {len(analyzers)} 个带标注病例")

    success = prior.learn_from_annotations(analyzers)
    if success and save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        prior.save_prior_knowledge(save_path)

    return prior


def apply_prior_to_patients(prior_path, data_dir, roi_dir, output_dir,
                           arterial_weight=2.0, min_cluster_fraction=0.05):
    print("\n" + "=" * 70)
    print("阶段2: 应用先验知识（强制3类 + HU梯度排序）")
    print("=" * 70)

    prior = TumorHeterogeneityPriorKnowledge(min_cluster_fraction=min_cluster_fraction)
    if not prior.load_prior_knowledge(prior_path):
        print("❌ 无法加载先验知识")
        return

    os.makedirs(output_dir, exist_ok=True)
    data_path = Path(data_dir)
    results = []

    print("\n🔍 处理所有患者...")
    for a_file in sorted((data_path / "A").glob("*.nii*")):
        name = a_file.name
        v_file = data_path / "V" / name
        roi_file = Path(roi_dir) / name

        out_name = safe_output_name(name)
        output_path = Path(output_dir) / out_name
        if output_path.exists():
            print(f"⏭️  跳过已处理: {name}")
            continue

        if not (v_file.exists() and roi_file.exists()):
            continue

        print(f"\n{'='*70}\n🔍 {name}")
        try:
            A_img, V_img, ROI_img, A_raw, V_raw, ROI, _, _, _ = load_nifti_data(a_file, v_file, roi_file)

            if np.sum(ROI) == 0:
                print("  ⚠️  ROI为空，跳过")
                continue

            analyzer = PancreaticTumorHeterogeneityAnalyzer(
                arterial_ct_raw=A_raw,
                venous_ct_raw=V_raw,
                tumor_mask=ROI,
                doctor_annotations=None,
                arterial_weight=arterial_weight,
                prior_knowledge=prior,
                min_cluster_fraction=min_cluster_fraction,
            )

            features = analyzer.extract_combined_features(use_raw_hu=True)
            hu_for_sort = features[:, 0]  # raw weighted HU
            labels_1d = analyzer.adaptive_clustering(features=features, method="prior_guided", hu_for_sort=hu_for_sort)
            analyzer.labels = labels_1d

            # 保存 3D 标签
            labels_3d = np.zeros(ROI.shape, dtype=np.uint16)
            labels_3d[ROI] = labels_1d.astype(np.uint16)
            nib.save(nib.Nifti1Image(labels_3d, A_img.affine), str(output_path))

            # 评估/描述符
            quality = analyzer.evaluate_clustering_quality(labels_1d, hu_for_sort=hu_for_sort)
            descriptors = analyzer.compute_tumor_descriptors(A_img, labels_1to3=labels_1d)

            print("\n📈 质量:")
            print(f"  簇数: {quality['n_clusters']} {'✓' if quality['exactly_3_clusters'] else '❌'}")
            print(f"  最小簇: {quality['min_cluster_fraction']*100:.1f}% | 约束: {'✓' if quality['min_constraint_satisfied'] else '✗'}")
            print(f"  HU梯度: {'✓' if quality['hu_gradient_correct'] else '❌'}")
            print(f"  平衡度: {quality['size_balance']:.3f}")
            print("  HU均值: " + " ".join([f"L{lb}={quality['hu_means'][lb]:.2f}" for lb in sorted(quality["hu_means"].keys())]))

            results.append({"case": name, **quality, **descriptors})
            print(f"\n✅ 已保存: {output_path.name}")

        except Exception as e:
            print(f"\n❌ 失败: {e}")
            import traceback
            traceback.print_exc()

    if results:
        df = pd.DataFrame(results)
        csv_path = Path(output_dir) / "results_summary.csv"
        df.to_csv(csv_path, index=False, encoding="utf-8-sig")
        print(f"\n{'='*70}")
        print(f"📊 汇总已保存: {csv_path}")

        invalid_cases = df[df["n_clusters"] != 3]
        gradient_incorrect = df[df["hu_gradient_correct"] == False]
        constraint_failed = df[df["min_constraint_satisfied"] == False]

        print("\n📈 统计:")
        print(f"  总病例数: {len(df)}")
        print(f"  3类病例: {len(df[df['n_clusters'] == 3])}/{len(df)}")
        print(f"  HU梯度正确: {len(df[df['hu_gradient_correct'] == True])}/{len(df)}")
        print(f"  约束满足: {len(df[df['min_constraint_satisfied'] == True])}/{len(df)}")

        if len(invalid_cases) > 0:
            print(f"\n⚠️  {len(invalid_cases)}个病例类别数异常:")
            for _, row in invalid_cases.iterrows():
                print(f"    {row['case']}: {row['n_clusters']}类")

        if len(gradient_incorrect) > 0:
            print(f"\n⚠️  {len(gradient_incorrect)}个病例HU梯度异常")

        if len(constraint_failed) > 0:
            print(f"\n⚠️  {len(constraint_failed)}个病例未满足{min_cluster_fraction*100:.0f}%约束")

        if len(invalid_cases) == 0 and len(gradient_incorrect) == 0 and len(constraint_failed) == 0:
            print(f"\n✅ 所有{len(df)}个病例均满足：3类 + HU梯度正确 + {min_cluster_fraction*100:.0f}%约束")


# ============================================================================
#  主函数
# ============================================================================

if __name__ == "__main__":
    DATA_DIR = "./ICC/data"

    ROI_DIR1 = "./ICC/data/inter/ROI/ROI1"
    ROI_DIR2 = "./ICC/data/inter/ROI/ROI2"

    DOT_DIR1 = "./ICC/data/inter/DOT/DOT1"
    DOT_DIR2 = "./ICC/data/inter/DOT/DOT2"

    OUTPUT_DIR = "./ICC/cluster/inter"
    os.makedirs(os.path.join(OUTPUT_DIR, "DOT1_ROI1"), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, "DOT2_ROI1"), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, "DOT1_ROI2"), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, "DOT2_ROI2"), exist_ok=True)

    PRIOR_PATH1 = "./ICC/inter/prior_knowledge_DOT1.pkl"
    PRIOR_PATH2 = "./ICC/inter/prior_knowledge_DOT2.pkl"

    ARTERIAL_WEIGHT = 2.0
    MIN_CLUSTER_FRACTION = 0.05

    # 阶段1: 学习先验知识
    # learn_prior_knowledge(DATA_DIR, ROI_DIR1, DOT_DIR1, ARTERIAL_WEIGHT, MIN_CLUSTER_FRACTION, PRIOR_PATH1)
    # learn_prior_knowledge(DATA_DIR, ROI_DIR1, DOT_DIR2, ARTERIAL_WEIGHT, MIN_CLUSTER_FRACTION, PRIOR_PATH2)

    # 阶段2: 应用先验知识进行聚类
    # apply_prior_to_patients(PRIOR_PATH1, DATA_DIR, ROI_DIR1, os.path.join(OUTPUT_DIR, "DOT1_ROI1"),
    #                         ARTERIAL_WEIGHT, MIN_CLUSTER_FRACTION)
    apply_prior_to_patients(PRIOR_PATH1, DATA_DIR, ROI_DIR2, os.path.join(OUTPUT_DIR, "DOT1_ROI2"),
                            ARTERIAL_WEIGHT, MIN_CLUSTER_FRACTION)
    # apply_prior_to_patients(PRIOR_PATH2, DATA_DIR, ROI_DIR1, os.path.join(OUTPUT_DIR, "DOT2_ROI1"),
    #                         ARTERIAL_WEIGHT, MIN_CLUSTER_FRACTION)
    apply_prior_to_patients(PRIOR_PATH2, DATA_DIR, ROI_DIR2, os.path.join(OUTPUT_DIR, "DOT2_ROI2"),
                            ARTERIAL_WEIGHT, MIN_CLUSTER_FRACTION)


阶段1: 学习先验知识（强制3类 + HU梯度排序）

🔍 扫描标注数据...
   BI HAI FENG_SXA.nii.gz A_max: 686.0 V_max: 606.0 ROI_max: 1.0 ANN_max: 3.0
  ✓ BI HAI FENG_SXA.nii.gz: 33个标注
   CAI LI FANG_TJA.nii.gz A_max: 1045.0 V_max: 858.0 ROI_max: 1.0 ANN_max: 3.0
  ✓ CAI LI FANG_TJA.nii.gz: 15个标注
   CAO QING_XYA.nii.gz A_max: 1395.0 V_max: 1188.0 ROI_max: 1.0 ANN_max: 3.0
  ✓ CAO QING_XYA.nii.gz: 52个标注
   CAO YAN QIN_TJA.nii.gz A_max: 1809.0 V_max: 1779.0 ROI_max: 1.0 ANN_max: 3.0
  ✓ CAO YAN QIN_TJA.nii.gz: 14个标注
   CENG DE LONG_TJA.nii.gz A_max: 1578.0 V_max: 1565.0 ROI_max: 1.0 ANN_max: 3.0
  ✓ CENG DE LONG_TJA.nii.gz: 14个标注
   CENG LING QI_TJA.nii.gz A_max: 1420.0 V_max: 1290.0 ROI_max: 1.0 ANN_max: 3.0
  ✓ CENG LING QI_TJA.nii.gz: 27个标注
   CENG TING_TJA.nii.gz A_max: 742.0 V_max: 837.0 ROI_max: 1.0 ANN_max: 3.0
  ✓ CENG TING_TJA.nii.gz: 22个标注
   CHAI HUA_TJA.nii.gz A_max: 2014.0 V_max: 1690.0 ROI_max: 1.0 ANN_max: 3.0
  ✓ CHAI HUA_TJA.nii.gz: 13个标注
   CHEN CAI_TJA.nii.gz A_max: 1217.0 V_max: 1186.0 ROI_max: 1.0 

In [4]:
# ============================================================================
#  主函数
# ============================================================================

if __name__ == "__main__":
    DATA_DIR = "./ICC/data"

    ROI_DIR1 = "./ICC/data/intra/ROI/ROIa"
    ROI_DIR2 = "./ICC/data/intra/ROI/ROIb"

    DOT_DIR1 = "./ICC/data/intra/DOT/DOTa"
    DOT_DIR2 = "./ICC/data/intra/DOT/DOTb"

    OUTPUT_DIR = "./ICC/cluster/intra"
    os.makedirs(os.path.join(OUTPUT_DIR, "DOT1_ROI1"), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, "DOT2_ROI1"), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, "DOT1_ROI2"), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, "DOT2_ROI2"), exist_ok=True)

    PRIOR_PATH1 = "./ICC/intra/prior_knowledge_DOTa.pkl"
    PRIOR_PATH2 = "./ICC/intra/prior_knowledge_DOTb.pkl"

    ARTERIAL_WEIGHT = 2.0
    MIN_CLUSTER_FRACTION = 0.05

    # 阶段1: 学习先验知识
    learn_prior_knowledge(DATA_DIR, ROI_DIR1, DOT_DIR1, ARTERIAL_WEIGHT, MIN_CLUSTER_FRACTION, PRIOR_PATH1)
    learn_prior_knowledge(DATA_DIR, ROI_DIR1, DOT_DIR2, ARTERIAL_WEIGHT, MIN_CLUSTER_FRACTION, PRIOR_PATH2)

    # 阶段2: 应用先验知识进行聚类
    apply_prior_to_patients(PRIOR_PATH1, DATA_DIR, ROI_DIR1, os.path.join(OUTPUT_DIR, "DOT1_ROI1"),
                            ARTERIAL_WEIGHT, MIN_CLUSTER_FRACTION)
    apply_prior_to_patients(PRIOR_PATH1, DATA_DIR, ROI_DIR2, os.path.join(OUTPUT_DIR, "DOT1_ROI2"),
                            ARTERIAL_WEIGHT, MIN_CLUSTER_FRACTION)
    apply_prior_to_patients(PRIOR_PATH2, DATA_DIR, ROI_DIR1, os.path.join(OUTPUT_DIR, "DOT2_ROI1"),
                            ARTERIAL_WEIGHT, MIN_CLUSTER_FRACTION)
    apply_prior_to_patients(PRIOR_PATH2, DATA_DIR, ROI_DIR2, os.path.join(OUTPUT_DIR, "DOT2_ROI2"),
                            ARTERIAL_WEIGHT, MIN_CLUSTER_FRACTION)


阶段1: 学习先验知识（强制3类 + HU梯度排序）

🔍 扫描标注数据...
   BI HAI FENG_SXA.nii.gz A_max: 686.0 V_max: 606.0 ROI_max: 1.0 ANN_max: 3.0
  ✓ BI HAI FENG_SXA.nii.gz: 33个标注
   CAI LI FANG_TJA.nii.gz A_max: 1045.0 V_max: 858.0 ROI_max: 1.0 ANN_max: 3.0
  ✓ CAI LI FANG_TJA.nii.gz: 15个标注
   CAO QING_XYA.nii.gz A_max: 1395.0 V_max: 1188.0 ROI_max: 1.0 ANN_max: 3.0
  ✓ CAO QING_XYA.nii.gz: 52个标注
   CAO YAN QIN_TJA.nii.gz A_max: 1809.0 V_max: 1779.0 ROI_max: 1.0 ANN_max: 3.0
  ✓ CAO YAN QIN_TJA.nii.gz: 14个标注
   CENG DE LONG_TJA.nii.gz A_max: 1578.0 V_max: 1565.0 ROI_max: 1.0 ANN_max: 3.0
  ✓ CENG DE LONG_TJA.nii.gz: 14个标注
   CENG LING QI_TJA.nii.gz A_max: 1420.0 V_max: 1290.0 ROI_max: 1.0 ANN_max: 3.0
  ✓ CENG LING QI_TJA.nii.gz: 27个标注
   CENG TING_TJA.nii.gz A_max: 742.0 V_max: 837.0 ROI_max: 1.0 ANN_max: 3.0
  ✓ CENG TING_TJA.nii.gz: 22个标注
   CHAI HUA_TJA.nii.gz A_max: 2014.0 V_max: 1690.0 ROI_max: 1.0 ANN_max: 3.0
  ✓ CHAI HUA_TJA.nii.gz: 13个标注
   CHEN CAI_TJA.nii.gz A_max: 1217.0 V_max: 1186.0 ROI_max: 1.0 

In [12]:
import os
import numpy as np
import nibabel as nib
import pandas as pd

# 计算dice系数
def dice_coefficient(y_true, y_pred):
    intersection = np.sum(y_true * y_pred)
    union = np.sum(y_true) + np.sum(y_pred)
    dice = (2.0 * intersection) / union
    return dice


df = pd.DataFrame()

for dataset in ['inter', 'intra']:

    OUTPUT_DIR = "./ICC/cluster/{}".format(dataset)
    path1_1 = os.path.join(OUTPUT_DIR, "DOT1_ROI1")
    path1_2 = os.path.join(OUTPUT_DIR, "DOT1_ROI2")
    path2_1 = os.path.join(OUTPUT_DIR, "DOT2_ROI1")
    path2_2 = os.path.join(OUTPUT_DIR, "DOT2_ROI2")

    # 计算dice系数
    dice_roi = []
    dice1_1 = []
    dice1_2 = []
    dice2_1 = []
    dice2_2 = []
    dice3_1 = []
    dice3_2 = []

    for file in os.listdir(path1_1):
        if file.endswith(".nii.gz"):
            # 读取文件
            mask1_1 = nib.load(os.path.join(path1_1, file)).get_fdata()
            mask1_2 = nib.load(os.path.join(path1_2, file)).get_fdata()
            mask2_1 = nib.load(os.path.join(path2_1, file)).get_fdata()
            mask2_2 = nib.load(os.path.join(path2_2, file)).get_fdata()

            # 计算dice系数
            # dice_roi.append(dice_coefficient(mask1_1>0, mask1_2>0))
            dice1_1.append(dice_coefficient(mask1_1 == 1, mask1_2 == 1))
            dice1_2.append(dice_coefficient(mask2_1 == 1, mask2_2 == 1))
            dice2_1.append(dice_coefficient(mask1_1 == 2, mask1_2 == 2))
            dice2_2.append(dice_coefficient(mask2_1 == 2, mask2_2 == 2))
            dice3_1.append(dice_coefficient(mask1_1 == 3, mask1_2 == 3))
            dice3_2.append(dice_coefficient(mask2_1 == 3, mask2_2 == 3))


            # # 对于类别i
            # for i in range(1, 4):
            #     # 计算dice系数
            #     dice1 = dice_coefficient(mask1_1 == i, mask1_2 == i)
            #     dice2 = dice_coefficient(mask2_1 == i, mask2_2 == i)
            #     dice3 = dice_coefficient(mask1_1 == i, mask2_1 == i)
            #     dice4 = dice_coefficient(mask1_2 == i, mask2_2 == i)

            #     print(f"for class {i} of {file}: Dice1 = {dice1:.4f}, Dice2 = {dice2:.4f}, Dice3 = {dice3:.4f}, Dice4 = {dice4:.4f}")

    print(f"for {dataset}: Dice1_1 = {np.mean(dice1_1):.4f}, Dice1_2 = {np.mean(dice1_2):.4f}, Dice2_1 = {np.mean(dice2_1):.4f}, Dice2_2 = {np.mean(dice2_2):.4f}, Dice3_1 = {np.mean(dice3_1):.4f}, Dice3_2 = {np.mean(dice3_2):.4f}")

    # print(f"for {dataset}: Dice_roi = {np.mean(dice_roi):.4f}")

for inter: Dice1_1 = 0.8196, Dice1_2 = 0.8197, Dice2_1 = 0.7814, Dice2_2 = 0.7830, Dice3_1 = 0.9021, Dice3_2 = 0.9021
for intra: Dice1_1 = 0.9172, Dice1_2 = 0.9172, Dice2_1 = 0.9108, Dice2_2 = 0.9107, Dice3_1 = 0.9715, Dice3_2 = 0.9709


In [26]:
# DDI 计算和可视化
import nibabel as nib
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import skew, kurtosis, entropy
import matplotlib.pyplot as plt
import os

def calculate_tumor_heterogeneity_score(image_path, cluster_path, bins=200, hu_range=(-160, 240)):
    """
    计算肿瘤异质性综合评分
    
    Parameters:
    -----------
    image_path : str
        HU值图像的nifti文件路径
    cluster_path : str  
        肿瘤聚类标签的nifti文件路径
    bins : int
        直方图bin数量
    hu_range : tuple
        HU值范围
        
    Returns:
    --------
    dict : 包含评分和详细信息的字典
    """
    
    # 1. 读取nifti文件
    image_nii = nib.load(image_path)
    cluster_nii = nib.load(cluster_path)
    
    image_data = image_nii.get_fdata()
    cluster_data = cluster_nii.get_fdata()
    
    # 2. 提取三个聚类的HU值
    cluster_hu_values = {}
    cluster_stats = {}
    histograms = {}
    
    for cluster_id in [1, 2, 3]:
        mask = cluster_data == cluster_id
        hu_values = image_data[mask]
        
        if len(hu_values) == 0:
            print(f"Warning: Cluster {cluster_id} is empty!")
            continue
            
        # 过滤HU值范围
        hu_values = hu_values[(hu_values >= hu_range[0]) & (hu_values <= hu_range[1])]
        
        cluster_hu_values[cluster_id] = hu_values
        
        # 🆕 扩展统计量
        cluster_stats[cluster_id] = {
            'count': len(hu_values),
            'mean': np.mean(hu_values),
            'std': np.std(hu_values),
            'median': np.median(hu_values),
            'p10': np.percentile(hu_values, 10),
            'p25': np.percentile(hu_values, 25),
            'p75': np.percentile(hu_values, 75),
            'p90': np.percentile(hu_values, 90),
            'skewness': skew(hu_values),      # 偏度
            'kurtosis': kurtosis(hu_values),   # 峰度
            'iqr': np.percentile(hu_values, 75) - np.percentile(hu_values, 25),  # 四分位距
        }
        
        # 计算直方图
        hist, bin_edges = np.histogram(hu_values, bins=bins, range=hu_range, density=True)
        histograms[cluster_id] = {'hist': hist, 'bins': bin_edges}
        
        # 🆕 计算直方图熵
        hist_prob = hist / (np.sum(hist) + 1e-8)
        hist_prob = hist_prob[hist_prob > 0]  # 去除零值
        cluster_stats[cluster_id]['entropy'] = entropy(hist_prob)
    
    # 3. 计算综合差异指数
    if len(cluster_stats) != 3:
        raise ValueError("需要恰好3个聚类!")
    
    # 提取数据
    n1, n2, n3 = [cluster_stats[i]['count'] for i in [1, 2, 3]]
    mean1, mean2, mean3 = [cluster_stats[i]['mean'] for i in [1, 2, 3]]
    std1, std2, std3 = [cluster_stats[i]['std'] for i in [1, 2, 3]]
    
    total_n = n1 + n2 + n3
    proportions = np.array([n1, n2, n3]) / total_n
    
    # =========================================================================
    # 🆕 A. 组间-组内方差比 (Between-Within Variance Ratio) - 代替 imbalance_index
    # =========================================================================
    # 思路：类似ANOVA的F统计量，直接衡量聚类的分离质量
    
    # 全局均值
    all_hu_values = np.concatenate([cluster_hu_values[1], 
                                     cluster_hu_values[2], 
                                     cluster_hu_values[3]])
    global_mean = np.mean(all_hu_values)
    
    # 组间平方和 (Between-cluster Sum of Squares)
    between_ss = sum([
        cluster_stats[i]['count'] * (cluster_stats[i]['mean'] - global_mean) ** 2
        for i in [1, 2, 3]
    ])
    
    # 组内平方和 (Within-cluster Sum of Squares)
    within_ss = sum([
        cluster_stats[i]['count'] * (cluster_stats[i]['std'] ** 2)
        for i in [1, 2, 3]
    ])
    
    # 组间-组内方差比
    bw_variance_ratio = between_ss / (within_ss + 1e-8)
    
    # 归一化到 [0, 1] 范围（使用 tanh 压缩）
    imbalance_index = np.tanh(bw_variance_ratio / 10)  # 调整系数以适应数据范围
    
    # =========================================================================
    # B. 分布中心分离度 (✅ 保持原有逻辑)
    # =========================================================================
    centers = np.array([mean1, mean2, mean3])
    center_distances = []
    for i in range(3):
        for j in range(i+1, 3):
            dist = abs(centers[i] - centers[j])
            weight = proportions[i] * proportions[j]
            center_distances.append(dist * weight)
    
    weighted_separation = np.mean(center_distances)
    max_possible_dist = hu_range[1] - hu_range[0]
    normalized_separation = weighted_separation / max_possible_dist
    
    # =========================================================================
    # C. 分布重叠度 (✅ 保持原有逻辑)
    # =========================================================================
    hist1, hist2, hist3 = [histograms[i]['hist'] for i in [1, 2, 3]]
    overlap_12 = np.sum(np.minimum(hist1, hist2)) * proportions[0] * proportions[1]
    overlap_13 = np.sum(np.minimum(hist1, hist3)) * proportions[0] * proportions[2] 
    overlap_23 = np.sum(np.minimum(hist2, hist3)) * proportions[1] * proportions[2]
    
    total_weight = np.sum([proportions[i] * proportions[j] for i in range(3) for j in range(i+1, 3)])
    weighted_overlap = (overlap_12 + overlap_13 + overlap_23) / (total_weight + 1e-8)
    separation_score = 1 - weighted_overlap
    
    # =========================================================================
    # 🆕 D. 分布形状异质性指数 (Shape Heterogeneity Index) - 代替 shape_diversity
    # =========================================================================
    # 思路：综合偏度、峰度、熵、IQR的差异，反映分布形状的多样性
    
    # 1. 偏度异质性 (Skewness Heterogeneity)
    skewness_values = np.array([cluster_stats[i]['skewness'] for i in [1, 2, 3]])
    skewness_heterogeneity = np.std(skewness_values)
    
    # 2. 峰度异质性 (Kurtosis Heterogeneity)
    kurtosis_values = np.array([cluster_stats[i]['kurtosis'] for i in [1, 2, 3]])
    kurtosis_heterogeneity = np.std(kurtosis_values)
    
    # 3. 熵异质性 (Entropy Heterogeneity)
    entropy_values = np.array([cluster_stats[i]['entropy'] for i in [1, 2, 3]])
    entropy_heterogeneity = np.std(entropy_values)
    
    # 4. IQR变异系数 (IQR Coefficient of Variation)
    iqr_values = np.array([cluster_stats[i]['iqr'] for i in [1, 2, 3]])
    iqr_cv = np.std(iqr_values) / (np.mean(iqr_values) + 1e-8)
    
    # 综合形状异质性（加权组合）
    shape_diversity = (
        0.25 * np.tanh(skewness_heterogeneity) +      # 偏度差异
        0.25 * np.tanh(kurtosis_heterogeneity / 2) +  # 峰度差异（除以2降低量级）
        0.30 * entropy_heterogeneity +                # 熵差异
        0.20 * np.tanh(iqr_cv * 2)                   # IQR变异系数
    )
    
    # =========================================================================
    # E. 综合异质性评分
    # =========================================================================
    heterogeneity_score = (
        0.20 * imbalance_index +           # 🆕 组间-组内方差比
        0.35 * normalized_separation +      # ✅ 中心分离度（保持）
        0.30 * separation_score +          # ✅ 重叠分离度（保持）
        0.15 * shape_diversity             # 🆕 分布形状异质性
    )
    
    # 4. 整理结果
    results = {
        'heterogeneity_score': heterogeneity_score,
        'components': {
            'imbalance_index': imbalance_index,           # 🆕 BW方差比
            'separation_index': normalized_separation,    # ✅ 保持
            'overlap_separation': separation_score,       # ✅ 保持
            'shape_diversity': shape_diversity            # 🆕 形状异质性
        },
        'detailed_metrics': {
            # 🆕 组间-组内方差相关
            'bw_variance_ratio': bw_variance_ratio,
            'between_ss': between_ss,
            'within_ss': within_ss,
            
            # 🆕 形状异质性相关
            'skewness_heterogeneity': skewness_heterogeneity,
            'kurtosis_heterogeneity': kurtosis_heterogeneity,
            'entropy_heterogeneity': entropy_heterogeneity,
            'iqr_cv': iqr_cv,
            
            # 各聚类的形状参数
            'skewness_values': skewness_values.tolist(),
            'kurtosis_values': kurtosis_values.tolist(),
            'entropy_values': entropy_values.tolist(),
            'iqr_values': iqr_values.tolist(),
            
            # 基本信息
            'proportions': proportions.tolist(),
            'centers': centers.tolist()
        },
        'cluster_stats': cluster_stats,
        'cluster_proportions': proportions.tolist(),
        'histograms': histograms
    }
    
    return results

def plot_cluster_distributions(results, save_path):
    """
    绘制三个聚类的HU值分布直方图，只保存不显示
    """
    plt.figure(figsize=(14, 8))
    
    colors = ['#FF9999', '#66B2FF', '#99FF99']
    cluster_names = ['Cluster 1', 'Cluster 2', 'Cluster 3']
    
    histograms = results['histograms']
    cluster_stats = results['cluster_stats']
    proportions = results['cluster_proportions']
    
    for i, cluster_id in enumerate([1, 2, 3]):
        if cluster_id in histograms:
            hist = histograms[cluster_id]['hist']
            bins = histograms[cluster_id]['bins']
            bin_centers = (bins[:-1] + bins[1:]) / 2
            
            n = cluster_stats[cluster_id]['count']
            prop = proportions[i]
            mean = cluster_stats[cluster_id]['mean']
            std = cluster_stats[cluster_id]['std']
            
            plt.fill_between(bin_centers, hist, alpha=0.7, color=colors[i], 
                          label=f'{cluster_names[i]} (n={n}, {prop*100:.1f}%, μ={mean:.1f}±{std:.1f})')
    
    plt.xlabel('HU Value', fontsize=13, fontweight='bold')
    plt.ylabel('Density', fontsize=13, fontweight='bold')
    
    # 标题显示组件评分
    components = results['components']
    title = (f"HU Value Distribution\n"
            f"Heterogeneity Score: {results['heterogeneity_score']:.3f} "
            f"(BWR={components['imbalance_index']:.3f}, "
            f"Sep={components['separation_index']:.3f}, "
            f"Overlap={components['overlap_separation']:.3f}, "
            f"Shape={components['shape_diversity']:.3f})")
    plt.title(title, fontsize=13, fontweight='bold')
    
    plt.legend(fontsize=10, loc='upper right')
    plt.grid(True, alpha=0.3, linestyle='--')
    
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()


# 使用示例
if __name__ == "__main__":
    
    for dataclass in ["inter", "intra"]:
        for dataset in ["A", "V"]:
            for cluster_type in ["DOT1_ROI1", "DOT1_ROI2", "DOT2_ROI1"]:
                result_df = pd.DataFrame()
                image_dir = f"./ICC/data/{dataset}"
                cluster_dir = f"./ICC/cluster/{dataclass}/{cluster_type}"
                csv_path = f"./ICC/result/{dataclass}/DDI_{dataset}_{cluster_type}.csv"

                namelist = os.listdir(image_dir)
                for name in namelist:
                    patient_id = name.split(".")[0]
                    print(f"处理患者: {patient_id}")
                    if not os.path.exists(f"{cluster_dir}/{patient_id}_clusters.nii.gz"):
                        continue

                    results = calculate_tumor_heterogeneity_score(
                        image_path=f"{image_dir}/{name}",
                        cluster_path=f"{cluster_dir}/{patient_id}_clusters.nii.gz"
                    )
                    
                    # 打印结果
                    print(f"\n=== Patient: {patient_id} ===")
                    print(f"肿瘤异质性评分: {results['heterogeneity_score']:.3f}")
                    print("\n各组件评分:")
                    for key, value in results['components'].items():
                        print(f"  {key}: {value:.3f}")
                    
                    print(f"\n详细指标:")
                    for key, value in results['detailed_metrics'].items():
                        if isinstance(value, (int, float)):
                            print(f"  {key}: {value:.3f}")
                    
                    print(f"\n聚类数量比例: {results['cluster_proportions']}")
                    
                    # 收集数据
                    dict_data = {
                        "patient": patient_id, 
                        "score": results['heterogeneity_score'],
                        # 'normalized_separation': results['detailed_metrics']['normalized_separation'],
                        'kurtosis_heterogeneity_normalized': results['detailed_metrics']['kurtosis_heterogeneity_normalized'], 
                        'between_ss': results['detailed_metrics']['between_ss'], 
                        'iqr_cv_normalized': results['detailed_metrics']['iqr_cv_normalized'], 
                        'normalized_separation': results['detailed_metrics']['normalized_separation'], 
                        'iqr_cv_normalized': results['detailed_metrics']['iqr_cv_normalized']
                        
                    }
                    # # 添加主要组件
                    # dict_data.update(results['components'])
                    # # 添加详细指标
                    # dict_data.update({
                    #     'bw_variance_ratio': results['detailed_metrics']['bw_variance_ratio'],
                    #     'skewness_heterogeneity': results['detailed_metrics']['skewness_heterogeneity'],
                    #     'kurtosis_heterogeneity': results['detailed_metrics']['kurtosis_heterogeneity'],
                    #     'entropy_heterogeneity': results['detailed_metrics']['entropy_heterogeneity'],
                    #     'iqr_cv': results['detailed_metrics']['iqr_cv'],
                    # })
                    
                    df0 = pd.DataFrame([dict_data])
                    result_df = pd.concat([result_df, df0], axis=0, ignore_index=True)

                # 保存结果CSV文件
                result_df.to_csv(csv_path, index=False)
                print(f"\n处理完成! 共处理 {len(result_df)} 个患者")
                print(f"结果已保存至: {csv_path}")

处理患者: BI HAI FENG_SXA

=== Patient: BI HAI FENG_SXA ===
肿瘤异质性评分: 0.377

各组件评分:
  imbalance_index: 0.023
  separation_index: 0.005
  overlap_separation: 0.940
  shape_diversity: 0.589

详细指标:
  bw_variance_ratio: 0.229
  between_ss: 1234292.841
  within_ss: 5384206.096
  skewness_heterogeneity: 0.699
  kurtosis_heterogeneity: 1.507
  entropy_heterogeneity: 0.491
  iqr_cv: 0.395

聚类数量比例: [0.05320721253325451, 0.05468519065917824, 0.8921075968075672]


KeyError: 'kurtosis_heterogeneity_normalized'

In [ ]:
import os
import pandas as pd
import nibabel as nib
import numpy as np
from scipy.stats import entropy
from collections import Counter

def calculate_spherical_spatial_features(cluster_path, voxel_size=(1.0, 1.0, 1.0)):
    """
    仅提取重要空间异质性特征
    """
    cluster_nii = nib.load(cluster_path)
    cluster_data = cluster_nii.get_fdata()
    if len(cluster_data.shape) == 4:
        cluster_data = cluster_data[:, :, :, 0]
    unique_labels = np.unique(cluster_data)
    valid_clusters = [int(label) for label in unique_labels if label != 0]
    if len(valid_clusters) < 2:
        raise ValueError(f"Need at least 2 clusters, found: {valid_clusters}")

    tumor_mask = cluster_data > 0
    tumor_coords = np.where(tumor_mask)
    tumor_coords_3d = np.array([tumor_coords[0], tumor_coords[1], tumor_coords[2]]).T
    tumor_coords_3d = tumor_coords_3d * np.array(voxel_size)
    tumor_centroid = np.mean(tumor_coords_3d, axis=0)

    cluster_spherical_features = {}
    for cluster_id in valid_clusters:
        coords = np.where(cluster_data == cluster_id)
        coords_3d = np.array([coords[0], coords[1], coords[2]]).T
        coords_3d = coords_3d * np.array(voxel_size)
        relative_coords = coords_3d - tumor_centroid
        x, y, z = relative_coords[:, 0], relative_coords[:, 1], relative_coords[:, 2]
        r = np.sqrt(x**2 + y**2 + z**2)
        theta = np.arctan2(y, x)
        theta[theta < 0] += 2 * np.pi
        phi = np.arccos(np.clip(z / (r + 1e-8), -1, 1))

        # 径向特征
        radial_features = {
            'mean_r': np.mean(r),
            'std_r': np.std(r),
            'r_concentration': np.exp(-np.std(r)/(np.mean(r) + 1e-8)),
        }
        # 方位角特征
        n_azimuth_sectors = 8
        theta_sector_width = 2 * np.pi / n_azimuth_sectors
        theta_sectors = []
        for i in range(n_azimuth_sectors):
            sector_start = i * theta_sector_width
            sector_end = (i + 1) * theta_sector_width
            count = np.sum((theta >= sector_start) & (theta < sector_end))
            theta_sectors.append(count)
        theta_sectors = np.array(theta_sectors)
        expected_theta = len(theta) / n_azimuth_sectors
        theta_uniformity = 1 - np.std(theta_sectors) / (expected_theta + 1e-8)
        theta_concentration = np.max(theta_sectors) / (np.mean(theta_sectors) + 1e-8)
        azimuth_features = {
            'theta_uniformity': max(0, theta_uniformity),
            'theta_concentration': min(theta_concentration, 3.0),
            'theta_sectors': theta_sectors,
        }
        # 极角特征
        n_polar_bands = 6
        phi_band_width = np.pi / n_polar_bands
        phi_bands = []
        for i in range(n_polar_bands):
            band_start = i * phi_band_width
            band_end = (i + 1) * phi_band_width
            count = np.sum((phi >= band_start) & (phi < band_end))
            phi_bands.append(count)
        phi_bands = np.array(phi_bands)
        phi_mean = np.mean(phi)
        upper_hemisphere = np.sum(phi < np.pi/2)
        lower_hemisphere = np.sum(phi > np.pi/2)
        hemisphere_bias = (upper_hemisphere + 1) / (lower_hemisphere + 1)
        polar_features = {
            'phi_mean': phi_mean,
            'hemisphere_bias': hemisphere_bias,
            'phi_bands': phi_bands,
        }
        # 空间模式判据
        if radial_features['r_concentration'] > 0.7:
            if azimuth_features['theta_uniformity'] > 0.8:
                spatial_pattern = 'central_spherical'
            else:
                spatial_pattern = 'central_concentrated'
        elif azimuth_features['theta_concentration'] > 2.0:
            spatial_pattern = 'directional'
        elif polar_features['hemisphere_bias'] > 2.0 or polar_features['hemisphere_bias'] < 0.5:
            spatial_pattern = 'hemispherical'
        else:
            spatial_pattern = 'complex_irregular'

        cluster_spherical_features[cluster_id] = {
            'radial_features': radial_features,
            'azimuth_features': azimuth_features,
            'polar_features': polar_features,
            'spatial_pattern': spatial_pattern,
            'raw_coords': {
                'r': r,
                'theta': theta,
                'phi': phi,
            }
        }

    cluster_list = list(valid_clusters)
    # 1. 径向分布差异
    radial_means = [cluster_spherical_features[c]['radial_features']['mean_r'] for c in cluster_list]
    radial_concentrations = [cluster_spherical_features[c]['radial_features']['r_concentration'] for c in cluster_list]
    radial_position_diversity = np.std(radial_means) / (np.mean(radial_means) + 1e-8)
    radial_shape_diversity = np.std(radial_concentrations)
    # 2. 方位角分布差异
    theta_uniformities = [cluster_spherical_features[c]['azimuth_features']['theta_uniformity'] for c in cluster_list]
    azimuth_uniformity_diversity = np.std(theta_uniformities)
    # 3. 极角分布差异
    phi_means = [cluster_spherical_features[c]['polar_features']['phi_mean'] for c in cluster_list]
    polar_position_diversity = np.std(phi_means) / (np.pi / 2)
    # 4. 半球差异
    upper_ratios = []
    for cluster_id in cluster_list:
        phi_data = cluster_spherical_features[cluster_id]['raw_coords']['phi']
        upper = np.sum(phi_data < np.pi/2)
        total = len(phi_data)
        upper_ratio = upper / total
        upper_ratios.append(upper_ratio)
    hemisphere_diversity = np.std(upper_ratios)
    # 5. 方位角集中度差异（Gini系数）
    def gini_coefficient(x):
        sorted_x = np.sort(x)
        n = len(x)
        cumsum = np.cumsum(sorted_x)
        return (2 * np.sum((n - np.arange(1, n+1) + 1) * sorted_x)) / (n * np.sum(sorted_x)) - (n + 1) / n
    gini_coefficients = []
    for cluster_id in cluster_list:
        theta_sectors = cluster_spherical_features[cluster_id]['azimuth_features']['theta_sectors']
        gini_coef = gini_coefficient(theta_sectors + 1)
        gini_coefficients.append(gini_coef)
    azimuth_concentration_diversity = np.std(gini_coefficients)
    # 6. 模式多样性 - Shannon熵
    patterns = [cluster_spherical_features[c]['spatial_pattern'] for c in cluster_list]
    pattern_counts = Counter(patterns)
    total_patterns = len(patterns)
    probs = np.array([count/total_patterns for count in pattern_counts.values()])
    shannon_entropy = entropy(probs, base=2)
    max_entropy = np.log2(len(pattern_counts)) if len(pattern_counts) > 1 else 0
    pattern_diversity = shannon_entropy / max_entropy if max_entropy > 0 else 0

    features = {
        'radial_position_diversity': radial_position_diversity,
        'radial_shape_diversity': radial_shape_diversity,
        'azimuth_uniformity_diversity': azimuth_uniformity_diversity,
        'azimuth_concentration_diversity': azimuth_concentration_diversity,
        'polar_position_diversity': polar_position_diversity,
        'hemisphere_diversity': hemisphere_diversity,
        'pattern_diversity': pattern_diversity
    }
    return features


def batch_process_clusters(cluster_dir, output_csv, voxel_size=(1.0, 1.0, 1.0)):
    """
    批量处理聚类文件并保存特征到CSV
    
    参数:
        cluster_dir: 聚类文件所在目录
        output_csv: 输出CSV文件路径
        voxel_size: 体素尺寸，默认(1.0, 1.0, 1.0)
    """
    # 获取所有聚类文件
    cluster_files = [f for f in os.listdir(cluster_dir) if f.endswith('.nii.gz')]
    
    if len(cluster_files) == 0:
        print(f"错误：在 {cluster_dir} 中未找到.nii.gz文件")
        return
    
    print(f"找到 {len(cluster_files)} 个聚类文件")
    print(f"{'='*80}")
    
    # 存储结果
    results_list = []
    
    # 逐个处理文件
    for i, file in enumerate(cluster_files, 1):
        try:
            print(f"\n[{i}/{len(cluster_files)}] 处理文件: {file}")
            
            # 提取患者ID
            patient_id = file.split('_clusters')[0]
            
            # 计算特征
            cluster_path = os.path.join(cluster_dir, file)
            features = calculate_spherical_spatial_features(cluster_path, voxel_size)
            
            # 添加患者ID
            features['patient_id'] = patient_id
            
            # 保存结果
            results_list.append(features)
            
            # 打印特征值
            print(f"  ✓ 成功提取特征:")
            print(f"    - radial_position_diversity: {features['radial_position_diversity']:.4f}")
            print(f"    - radial_shape_diversity: {features['radial_shape_diversity']:.4f}")
            print(f"    - azimuth_uniformity_diversity: {features['azimuth_uniformity_diversity']:.4f}")
            print(f"    - azimuth_concentration_diversity: {features['azimuth_concentration_diversity']:.4f}")
            print(f"    - polar_position_diversity: {features['polar_position_diversity']:.4f}")
            print(f"    - hemisphere_diversity: {features['hemisphere_diversity']:.4f}")
            print(f"    - pattern_diversity: {features['pattern_diversity']:.4f}")
            
        except Exception as e:
            print(f"  ✗ 处理失败: {str(e)}")
            import traceback
            traceback.print_exc()
            continue
    
    # 转换为DataFrame并保存
    if len(results_list) > 0:
        df = pd.DataFrame(results_list)
        
        # 调整列顺序，patient_id放在第一列
        cols = ['patient_id'] + [col for col in df.columns if col != 'patient_id']
        df = df[cols]
        
        # 保存到CSV
        df.to_csv(output_csv, index=False)
        
        print(f"\n{'='*80}")
        print(f"✅ 批量处理完成!")
        print(f"{'='*80}")
        print(f"  ✓ 成功处理: {len(results_list)}/{len(cluster_files)} 个文件")
        print(f"  ✓ 结果保存至: {output_csv}")
        print(f"  ✓ 特征维度: {len(df.columns)} 列")
        
        # 显示统计信息
        print(f"\n📊 特征统计摘要:")
        feature_cols = [col for col in df.columns if col != 'patient_id']
        print(df[feature_cols].describe().to_string())
        
    else:
        print(f"\n✗ 未能成功处理任何文件")


# ============================================================================
# 主程序调用
# ============================================================================

if __name__ == "__main__":   
    for dataclass in ["inter", "intra"]:
        for cluster_type in ["DOT1_ROI1", "DOT1_ROI2", "DOT2_ROI1"]:

            print(f"\n{'#'*80}")
            print(f"# 处理 Cluster Type {cluster_type}")
            print(f"{'#'*80}\n")
            
            # 输入输出路径
            cluster_dir = f"./ICC/cluster/{dataclass}/{cluster_type}"
            output_csv = f'./ICC/result/{dataclass}/SHI_{cluster_type}.csv'
            
            # 批量处理
            batch_process_clusters(
                cluster_dir=cluster_dir,
                output_csv=output_csv,
                voxel_size=(1.0, 1.0, 1.0)  # 根据实际情况调整体素尺寸
            )
        
        print(f"\n{'#'*80}")
        print(f"# 全部处理完成!")
        print(f"{'#'*80}")


################################################################################
# 处理 Cluster Type DOT1_ROI1
################################################################################

找到 30 个聚类文件

[1/30] 处理文件: BI HAI FENG_SXA_clusters.nii.gz
  ✓ 成功提取特征:
    - radial_position_diversity: 0.1039
    - radial_shape_diversity: 0.0718
    - azimuth_uniformity_diversity: 0.1514
    - azimuth_concentration_diversity: 0.0793
    - polar_position_diversity: 0.0945
    - hemisphere_diversity: 0.1017
    - pattern_diversity: 0.9183

[2/30] 处理文件: CAI LI FANG_TJA_clusters.nii.gz
  ✓ 成功提取特征:
    - radial_position_diversity: 0.1133
    - radial_shape_diversity: 0.0649
    - azimuth_uniformity_diversity: 0.1331
    - azimuth_concentration_diversity: 0.0724
    - polar_position_diversity: 0.2773
    - hemisphere_diversity: 0.2342
    - pattern_diversity: 0.0000

[3/30] 处理文件: CAO QING_XYA_clusters.nii.gz
  ✓ 成功提取特征:
    - radial_position_diversity: 0.0783
    - radial_shape_diversity: 0.0214
    

In [ ]:
# DDI 计算（仅特征计算，无可视化，无线性组合评分）
import nibabel as nib
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import skew, kurtosis, entropy
import os

def calculate_tumor_heterogeneity_features(image_path, cluster_path, bins=200, hu_range=(-160, 240)):
    """
    计算肿瘤异质性特征（不包含线性组合评分）
    
    Parameters:
    -----------
    image_path : str
        HU值图像的nifti文件路径
    cluster_path : str  
        肿瘤聚类标签的nifti文件路径
    bins : int
        直方图bin数量
    hu_range : tuple
        HU值范围
        
    Returns:
    --------
    dict : 包含原始特征和详细信息的字典
    """
    
    # 1. 读取nifti文件
    image_nii = nib.load(image_path)
    cluster_nii = nib.load(cluster_path)
    
    image_data = image_nii.get_fdata()
    cluster_data = cluster_nii.get_fdata()
    
    # 2. 提取三个聚类的HU值
    cluster_hu_values = {}
    cluster_stats = {}
    histograms = {}
    
    for cluster_id in [1, 2, 3]:
        mask = cluster_data == cluster_id
        hu_values = image_data[mask]
        
        if len(hu_values) == 0:
            print(f"Warning: Cluster {cluster_id} is empty!")
            continue
            
        # 过滤HU值范围
        hu_values = hu_values[(hu_values >= hu_range[0]) & (hu_values <= hu_range[1])]
        
        cluster_hu_values[cluster_id] = hu_values
        
        # 扩展统计量
        cluster_stats[cluster_id] = {
            'count': len(hu_values),
            'mean': np.mean(hu_values),
            'std': np.std(hu_values),
            'median': np.median(hu_values),
            'p10': np.percentile(hu_values, 10),
            'p25': np.percentile(hu_values, 25),
            'p75': np.percentile(hu_values, 75),
            'p90': np.percentile(hu_values, 90),
            'skewness': skew(hu_values),      # 偏度
            'kurtosis': kurtosis(hu_values),   # 峰度
            'iqr': np.percentile(hu_values, 75) - np.percentile(hu_values, 25),  # 四分位距
        }
        
        # 计算直方图
        hist, bin_edges = np.histogram(hu_values, bins=bins, range=hu_range, density=True)
        histograms[cluster_id] = {'hist': hist, 'bins': bin_edges}
        
        # 计算直方图熵
        hist_prob = hist / (np.sum(hist) + 1e-8)
        hist_prob = hist_prob[hist_prob > 0]  # 去除零值
        cluster_stats[cluster_id]['entropy'] = entropy(hist_prob)
    
    # 3. 计算特征指标
    if len(cluster_stats) != 3:
        raise ValueError("需要恰好3个聚类!")
    
    # 提取数据
    n1, n2, n3 = [cluster_stats[i]['count'] for i in [1, 2, 3]]
    mean1, mean2, mean3 = [cluster_stats[i]['mean'] for i in [1, 2, 3]]
    std1, std2, std3 = [cluster_stats[i]['std'] for i in [1, 2, 3]]
    
    total_n = n1 + n2 + n3
    proportions = np.array([n1, n2, n3]) / total_n
    
    # =========================================================================
    # A. 组间-组内方差比 (Between-Within Variance Ratio)
    # =========================================================================
    all_hu_values = np.concatenate([cluster_hu_values[1], 
                                     cluster_hu_values[2], 
                                     cluster_hu_values[3]])
    global_mean = np.mean(all_hu_values)
    
    # 组间平方和 (Between-cluster Sum of Squares)
    between_ss = sum([
        cluster_stats[i]['count'] * (cluster_stats[i]['mean'] - global_mean) ** 2
        for i in [1, 2, 3]
    ])
    
    # 组内平方和 (Within-cluster Sum of Squares)
    within_ss = sum([
        cluster_stats[i]['count'] * (cluster_stats[i]['std'] ** 2)
        for i in [1, 2, 3]
    ])
    
    # 组间-组内方差比（原始值）
    bw_variance_ratio = between_ss / (within_ss + 1e-8)
    
    # 归一化版本
    bw_variance_ratio_normalized = np.tanh(bw_variance_ratio / 10)
    
    # =========================================================================
    # B. 分布中心分离度
    # =========================================================================
    centers = np.array([mean1, mean2, mean3])
    center_distances = []
    for i in range(3):
        for j in range(i+1, 3):
            dist = abs(centers[i] - centers[j])
            weight = proportions[i] * proportions[j]
            center_distances.append(dist * weight)
    
    weighted_separation = np.mean(center_distances)
    max_possible_dist = hu_range[1] - hu_range[0]
    normalized_separation = weighted_separation / max_possible_dist
    
    # =========================================================================
    # C. 分布重叠度
    # =========================================================================
    hist1, hist2, hist3 = [histograms[i]['hist'] for i in [1, 2, 3]]
    overlap_12 = np.sum(np.minimum(hist1, hist2)) * proportions[0] * proportions[1]
    overlap_13 = np.sum(np.minimum(hist1, hist3)) * proportions[0] * proportions[2] 
    overlap_23 = np.sum(np.minimum(hist2, hist3)) * proportions[1] * proportions[2]
    
    total_weight = np.sum([proportions[i] * proportions[j] for i in range(3) for j in range(i+1, 3)])
    weighted_overlap = (overlap_12 + overlap_13 + overlap_23) / (total_weight + 1e-8)
    overlap_separation_score = 1 - weighted_overlap
    
    # =========================================================================
    # D. 分布形状异质性的各个子组件
    # =========================================================================
    # 1. 偏度异质性 (Skewness Heterogeneity)
    skewness_values = np.array([cluster_stats[i]['skewness'] for i in [1, 2, 3]])
    skewness_heterogeneity = np.std(skewness_values)
    skewness_heterogeneity_normalized = np.tanh(skewness_heterogeneity)
    
    # 2. 峰度异质性 (Kurtosis Heterogeneity)
    kurtosis_values = np.array([cluster_stats[i]['kurtosis'] for i in [1, 2, 3]])
    kurtosis_heterogeneity = np.std(kurtosis_values)
    kurtosis_heterogeneity_normalized = np.tanh(kurtosis_heterogeneity / 2)
    
    # 3. 熵异质性 (Entropy Heterogeneity)
    entropy_values = np.array([cluster_stats[i]['entropy'] for i in [1, 2, 3]])
    entropy_heterogeneity = np.std(entropy_values)
    
    # 4. IQR变异系数 (IQR Coefficient of Variation)
    iqr_values = np.array([cluster_stats[i]['iqr'] for i in [1, 2, 3]])
    iqr_cv = np.std(iqr_values) / (np.mean(iqr_values) + 1e-8)
    iqr_cv_normalized = np.tanh(iqr_cv * 2)
    
    # 4. 整理结果（不包含线性组合的总分）
    results = {
        # 四大核心特征（归一化后的版本，可直接用于线性组合）
        'features': {
            'bw_variance_ratio_normalized': bw_variance_ratio_normalized,  # 组间-组内方差比（归一化）
            'normalized_separation': normalized_separation,                 # 中心分离度
            'overlap_separation_score': overlap_separation_score,          # 重叠分离度
            'skewness_heterogeneity_normalized': skewness_heterogeneity_normalized,  # 偏度异质性（归一化）
            'kurtosis_heterogeneity_normalized': kurtosis_heterogeneity_normalized,  # 峰度异质性（归一化）
            'entropy_heterogeneity': entropy_heterogeneity,                # 熵异质性
            'iqr_cv_normalized': iqr_cv_normalized,                        # IQR变异系数（归一化）
        },
        
        # 原始数值（未归一化）
        'raw_metrics': {
            'bw_variance_ratio': bw_variance_ratio,
            'between_ss': between_ss,
            'within_ss': within_ss,
            'weighted_separation': weighted_separation,
            'weighted_overlap': weighted_overlap,
            'skewness_heterogeneity': skewness_heterogeneity,
            'kurtosis_heterogeneity': kurtosis_heterogeneity,
            'entropy_heterogeneity': entropy_heterogeneity,
            'iqr_cv': iqr_cv,
        },
        
        # 各聚类的统计参数
        'cluster_parameters': {
            'skewness_values': skewness_values.tolist(),
            'kurtosis_values': kurtosis_values.tolist(),
            'entropy_values': entropy_values.tolist(),
            'iqr_values': iqr_values.tolist(),
            'proportions': proportions.tolist(),
            'centers': centers.tolist(),
            'stds': [std1, std2, std3],
        },
        
        # 详细聚类统计信息
        'cluster_stats': cluster_stats,
        'cluster_proportions': proportions.tolist(),
        'histograms': histograms
    }
    
    return results


# 批量处理主程序
if __name__ == "__main__":   
    for dataclass in ["inter", "intra"]:
        for cluster_type in ["DOT1_ROI1", "DOT1_ROI2", "DOT2_ROI1"]:
            
            result_df = pd.DataFrame()
            
            arterial_dir =  f"./ICC/data/A"
            venous_dir = f"./ICC/data/V"
            cluster_dir = f'./ICC/cluster/{dataclass}/{cluster_type}'
            csv_path = f"./ICC/result/{dataclass}/DEHI_{cluster_type}.csv"
            
            cluster_files = [f for f in os.listdir(cluster_dir) if f.endswith('.nii.gz')]
            
            print(f"Found {len(cluster_files)} cluster files to process in '{cluster_dir}'")
            
            for i, cluster_file in enumerate(cluster_files):
                print(f"\n{'='*60}\nProcessing file {i+1}/{len(cluster_files)}: {cluster_file}")
                
                patient_id = cluster_file.split('_clusters')[0]
                base_filename = f"{patient_id}.nii.gz"
                
                arterial_path = os.path.join(arterial_dir, base_filename)
                venous_path = os.path.join(venous_dir, base_filename)
                cluster_path = os.path.join(cluster_dir, cluster_file)
                
                # 检查所有文件是否存在
                if not all(os.path.exists(p) for p in [arterial_path, venous_path, cluster_path]):
                    print(f"⚠️ Skipping {patient_id}: One or more files are missing.")
                    print(f"  Arterial: {arterial_path} (Exists: {os.path.exists(arterial_path)})")
                    print(f"  Venous:   {venous_path} (Exists: {os.path.exists(venous_path)})")
                    print(f"  Cluster:  {cluster_path} (Exists: {os.path.exists(cluster_path)})")
                    continue

                try:
                    # 计算变化异质性
                    results = calculate_change_heterogeneity(
                        arterial_path=arterial_path,
                        venous_path=venous_path,
                        cluster_path=cluster_path,
                        hu_range=(-160, 240)
                    )
                    
                    # 提取特征到DataFrame
                    df_row = extract_change_features_to_dataframe(results, patient_id)
                    result_df = pd.concat([result_df, df_row], ignore_index=True)
                    
                    print(f"✓ Successfully processed {patient_id}")
                    print(f"  - Change Heterogeneity Index: {results['change_heterogeneity_index']:.4f}")
                    print(f"  - Overall Mean Change (A-V): {results['summary_stats']['overall_mean_change']:.1f} HU")

                except Exception as e:
                    print(f"❌ Error processing {patient_id}: {e}")

            # 保存最终的特征表格
            result_df.to_csv(csv_path, index=False)

Found 30 cluster files to process in './ICC/cluster/inter/DOT1_ROI1'

Processing file 1/30: BI HAI FENG_SXA_clusters.nii.gz
Loading multi-phase CT data...
Arterial shape: (170, 91, 99)
Venous shape: (170, 91, 99)
Cluster shape: (170, 91, 99)
HU Change (A-V) range: -1062.0 to 1041.0 HU
Valid clusters: [1, 2, 3]

Analyzing cluster 1...
  Mean Change (A-V): -3.4 ± 81.4 HU
  Dominant pattern: anomalous_enhancement
  Phase correlation: 0.876

Analyzing cluster 2...
  Mean Change (A-V): 55.4 ± 92.2 HU
  Dominant pattern: significant_weakening
  Phase correlation: -0.769

Analyzing cluster 3...
  Mean Change (A-V): 46.1 ± 78.3 HU
  Dominant pattern: significant_weakening
  Phase correlation: 0.232
✓ Successfully processed BI HAI FENG_SXA
  - Change Heterogeneity Index: 3.3052
  - Overall Mean Change (A-V): 32.7 HU

Processing file 2/30: CAI LI FANG_TJA_clusters.nii.gz
Loading multi-phase CT data...
Arterial shape: (141, 81, 94)
Venous shape: (141, 81, 94)
Cluster shape: (141, 81, 94)
HU Chang

In [ ]:
# 读取数据
ddi_a = pd.read_csv('./ICC/result/inter/DDI_A_DOT1_ROI1.csv')
# 在col中加上A_
ddi_a.columns = ['A_' + col for col in ddi_a.columns]
ddi_v = pd.read_csv('./ICC/result/inter/DDI_V_DOT1_ROI1.csv')
# 在col中加上V_
ddi_v.columns = ['V_' + col for col in ddi_v.columns]

shi = pd.read_csv('./ICC/result/inter/SHI_DOT1_ROI1.csv')
dehi = pd.read_csv('./ICC/result/inter/DEHI_DOT1_ROI1.csv')

# 合并数据
merged_df = pd.concat([ddi_a, ddi_v, shi, dehi], axis=0)
col = ['change_magnitude_diversity', 'A_normalized_separation', 'A_kurtosis_heterogeneity_normalized', 
       'radial_shape_diversity', 'sig_weakening_ratio_avg', 'V_between_ss', 'stable_ratio_avg',
       'V_iqr_cv_normalized','V_normalized_separation', 'A_iqr_cv_normalized']
weight = [2.128347, -1.285431, 1.134319, -0.656003, -0.482377, 0.367101, 0.287913, -0.099048, -0.096684, -0.060550]

# 计算加权得分
merged_df['weighted_score'] = merged_df[col].dot(weight)
print(merged_df['weighted_score'])

KeyError: "['A_normalized_separation', 'A_kurtosis_heterogeneity_normalized', 'V_between_ss', 'V_iqr_cv_normalized', 'V_normalized_separation', 'A_iqr_cv_normalized'] not in index"

In [36]:
import numpy as np
import pandas as pd
from scipy.stats import f

def icc_two_arrays_test(
    pairs: dict,                 # {feature: (x, y)} 其中 x/y 是 array-like 或 Series
    alpha: float = 0.05,
    nan_policy: str = "drop"     # "drop" | "raise"
) -> pd.DataFrame:
    rows = []
    for feat, (x, y) in pairs.items():
        X = pd.DataFrame({"col1": x, "col2": y})

        if X.isna().any().any():
            if nan_policy == "raise":
                raise ValueError(f"{feat} has NaNs")
            X = X.dropna(axis=0)

        Y = X.to_numpy(float)
        n, k = Y.shape  # k=2
        if n < 2:
            rows.append({"feature": feat, "ICC(3,1)": np.nan, "p": np.nan, "n": n})
            continue

        mean_row = Y.mean(axis=1, keepdims=True)
        mean_col = Y.mean(axis=0, keepdims=True)
        grand = Y.mean()

        ss_total = ((Y - grand) ** 2).sum()
        ss_rows  = (k * ((mean_row - grand) ** 2)).sum()
        ss_cols  = (n * ((mean_col - grand) ** 2)).sum()
        ss_err   = ss_total - ss_rows - ss_cols

        msr = ss_rows / (n - 1)
        mse = ss_err / ((n - 1) * (k - 1))

        icc = (msr - mse) / (msr + (k - 1) * mse)

        Fv = msr / mse
        df1, df2 = n - 1, (n - 1) * (k - 1)  # n-1
        p = 1 - f.cdf(Fv, df1, df2)

        Fl = f.ppf(alpha / 2, df1, df2)
        Fu = f.ppf(1 - alpha / 2, df1, df2)
        ci_low  = (Fv / Fu - 1) / (Fv / Fu + (k - 1))
        ci_high = (Fv / Fl - 1) / (Fv / Fl + (k - 1))

        rows.append({
            "feature": feat,
            "ICC(3,1)": icc,
            "F": Fv, "df1": df1, "df2": df2,
            "p": p,
            f"CI{int((1-alpha)*100)}_low": ci_low,
            f"CI{int((1-alpha)*100)}_high": ci_high,
            "n": n
        })

    return pd.DataFrame(rows).sort_values("p").reset_index(drop=True)


# 计算icc
data1 = pd.read_csv("./ICC/result/inter/DEHI_DOT1_ROI1.csv")
data2 = pd.read_csv("./ICC/result/inter/DEHI_DOT1_ROI2.csv")
data3 = pd.read_csv("./ICC/result/inter/DEHI_DOT2_ROI1.csv")

pairs = {
    "intra": (data1["score"], data2["score"]),
    "inter": (data1["score"], data3["score"])
}

res = icc_two_arrays_test(pairs)
print(res)

  feature  ICC(3,1)            F  df1  df2             p  CI95_low  CI95_high  \
0   inter  0.999219  2561.083343   29   29  1.110223e-16  0.998361   0.999628   
1   intra  0.642887     4.600474   29   29  4.805237e-05  0.372975   0.812481   

    n  
0  30  
1  30  
